In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import tool


import requests
import math


In [12]:
load_dotenv()

True

In [14]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002D9CE5803E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002D9CE580770>, root_client=<openai.OpenAI object at 0x000002D9CE580180>

In [15]:
# Tools
search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

@tool
def calculator(expression:str)-> str:
    """Useful for simple math calculations"""
    try:
        allowed = {
            "math": math,
            "abs": abs,
            "round": round,
            "min": min,
            "max": max,
            "sum": sum
        }

        result = eval(expression, {"__builtin__": {}}, allowed)
        return str(result)

    except Exception as e:
        print(f"Error in calculation: {str(e)}")


def get_stock_price(symbol: str)-> dict:
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=DHUVDLZVG9P6TN1H"
    r = requests.get(url)
    return r.json()


In [16]:
#Make tool list
tools = [get_stock_price, search_tool, calculator]

#make the llm tool aware 
llm_with_tools = llm.bind_tools(tools)



In [20]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [27]:
def chat_node(state: ChatState):
    """LLM node that answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

tool_node = ToolNode(tools)

ValueError: Function must have a docstring if description not provided.

In [26]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges("chat_node", tools_condition)
#add_edges

# graph.add_edge("tools", "chat_node")



NameError: name 'tool_node' is not defined

In [ ]:
chatbot = graph.compile()
chatbot

In [ ]:
out = chatbot.invoke({"messages": [HumanMessage(coontent="Hellow oerdl")]})
print(out["messages"][-1].content)